In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score
import plotly.express as px
import plotly.graph_objects as go
import os

# ==============================================================================
# PASSO 1: LIMPEZA E PREPARAÇÃO DOS DADOS BRUTOS DO IBGE(SIDRA)
# Esta etapa é a Base de Dados e Pré-Processamento
# ==============================================================================
print(">>> [FASE 1 de 6] Carregando e limpando os 3 arquivos do IBGE...")

NOME_DA_PASTA = 'dados_brutos'

# --- Construção dos caminhos para os arquivos ---
caminho_pib = os.path.join(NOME_DA_PASTA, 'pib.xlsx')
caminho_alfabetizacao = os.path.join(NOME_DA_PASTA, 'alfabetizacao.xlsx')
caminho_urbanizacao = os.path.join(NOME_DA_PASTA, 'urbanizacao.xlsx')

# --- Limpeza do arquivo de PIB ---
# O arquivo vem com cabeçalhos e rodapés que precisam ser removidos.
df_pib = pd.read_excel(caminho_pib, sep=';', skiprows=3, encoding='latin1', skipfooter=3, engine='python')
df_pib = df_pib.iloc[:, [0, -1]]
df_pib.columns = ['codigo_ibge', 'pib_per_capita']
df_pib['pib_per_capita'] = pd.to_numeric(df_pib['pib_per_capita'], errors='coerce')

# --- Limpeza do arquivo de ALFABETIZAÇÃO ---
df_alfabetizacao = pd.read_excel(caminho_alfabetizacao, sep=';', skiprows=3, encoding='latin1', skipfooter=3, engine='python')
df_alfabetizacao = df_alfabetizacao.iloc[:, [0, -1]]
df_alfabetizacao.columns = ['codigo_ibge', 'taxa_alfabetizacao']
df_alfabetizacao['taxa_alfabetizacao'] = pd.to_numeric(df_alfabetizacao['taxa_alfabetizacao'], errors='coerce')

# --- Limpeza do arquivo de URBANIZAÇÃO ---
df_urbanizacao = pd.read_excel(caminho_urbanizacao, sep=';', skiprows=4, encoding='latin1', skipfooter=3, engine='python')
# Esta tabela é diferente, precisamos calcular a taxa.
df_pop_urbana = df_urbanizacao[df_urbanizacao['Situação do domicílio'] == 'Urbana'].copy()
df_pop_total = df_urbanizacao.groupby('Município (Código)')['Valor'].sum().reset_index()
# Juntar os dois
df_taxa_urb = pd.merge(df_pop_urbana, df_pop_total, on='Município (Código)')
df_taxa_urb['taxa_urbanizacao'] = (df_taxa_urb['Valor_x'] / df_taxa_urb['Valor_y']) * 100
df_taxa_urb = df_taxa_urb[['Município (Código)', 'taxa_urbanizacao']].rename(columns={'Município (Código)': 'codigo_ibge'})

# --- Unificação dos dados em uma única tabela ---
print(">>> Unificando os dados...")
df_final = pd.merge(df_pib, df_alfabetizacao, on='codigo_ibge', how='inner')
df_final = pd.merge(df_final, df_taxa_urb, on='codigo_ibge', how='inner')
# Remove qualquer município que não tenha todos os dados
df_final = df_final.dropna()

print(f">>> Dados consolidados! Total de {len(df_final)} municípios com dados completos.")

: 

In [ ]:
# ==============================================================================
# PASSO 2: NORMALIZAÇÃO Z-SCORE
# Na da seção Metodologia
# Padronizamos as escalas para que PIB (milhares de R$) não domine a análise.
# ==============================================================================
print("\n>>> [FASE 2 de 6] Normalizando os dados com Z-Score...")
indicadores = ['pib_per_capita', 'taxa_alfabetizacao', 'taxa_urbanizacao']
scaler = StandardScaler()
dados_normalizados = scaler.fit_transform(df_final[indicadores])
df_normalizado = pd.DataFrame(dados_normalizados, columns=indicadores)

In [ ]:
# ==============================================================================
# PASSO 3: DETERMINAÇÃO DO NÚMERO DE CLUSTERS (K)
# Técnicas de Agrupamento
# ==============================================================================
print("\n>>> [FASE 3 de 6] Calculando o número ideal de perfis (K)...")

# --- 3.1: Método do Cotovelo ---
inercias = []
k_range = range(2, 11)
for k in k_range:
    modelo = KMedoids(n_clusters=k, random_state=42)
    modelo.fit(df_normalizado)
    inercias.append(modelo.inertia_)

fig_cotovelo = px.line(x=k_range, y=inercias, title='Método do Cotovelo',
                       labels={'x':'Número de Clusters (K)', 'y':'Inércia'}, markers=True)
fig_cotovelo.show()

# --- 3.2: Análise da Silhueta ---
silhuetas = []
for k in k_range:
    modelo = KMedoids(n_clusters=k, random_state=42)
    labels = modelo.fit_predict(df_normalizado)
    silhuetas.append(silhouette_score(df_normalizado, labels))

fig_silhueta = px.bar(x=k_range, y=silhuetas, title='Análise da Silhueta',
                      labels={'x':'Número de Clusters (K)', 'y':'Pontuação da Silhueta'})
fig_silhueta.show()

# --- Decisão do K ---
# O "cotovelo" e a maior barra da silhueta sugerem o melhor K.
K_ESCOLHIDO = 4 # Valor inicial.
print(f">>> Com base nos gráficos, vamos prosseguir com K = {K_ESCOLHIDO} perfis.")

In [ ]:
# ==============================================================================
# PASSO 4: APLICAÇÃO DO ALGORITMO K-MEDOID
# A Justificativa Técnica
# Robusto a outliers como São Paulo ou cidades muito pequenas.
# ==============================================================================
print(f"\n>>> [FASE 4 de 6] Segmentando os municípios em {K_ESCOLHIDO} perfis com K-Medoid...")
modelo_final = KMedoids(n_clusters=K_ESCOLHIDO, random_state=42)
clusters = modelo_final.fit_predict(df_normalizado)
df_final['cluster'] = clusters
df_final['cluster'] = df_final['cluster'].astype(str)

In [ ]:
# ==============================================================================
# PASSO 5: VISUALIZAÇÃO E INTERPRETAÇÃO DOS PERFIS
# É aqui que transformamos os dados em Inteligência Estratégica
# ==============================================================================
print("\n>>> [FASE 5 de 6] Gerando os gráficos finais para a apresentação...")

# --- 5.1: Gráfico de Radar ---
# Mostra a "personalidade" de cada grupo.
df_normalizado['cluster'] = clusters
df_radar = df_normalizado.groupby('cluster').mean().reset_index()
df_radar_melted = pd.melt(df_radar, id_vars=['cluster'], var_name='Indicador', value_name='Valor (Normalizado)')

fig_radar = px.line_polar(df_radar_melted, r='Valor (Normalizado)', theta='Indicador', color='cluster',
                          line_close=True, title='Perfil Socioeconômico de Cada Grupo de Municípios',
                          template='plotly_dark')
fig_radar.update_traces(fill='toself')
fig_radar.show()


# --- 5.2: Boxplots para cada indicador ---
# Prova visual de que os grupos são realmente diferentes.
for indicador in indicadores:
    fig_box = px.box(df_final, x='cluster', y=indicador, color='cluster',
                     title=f'Distribuição de "{indicador}" por Grupo', log_y=(indicador == 'pib_per_capita'))
    fig_box.show()

In [ ]:
# ==============================================================================
# PASSO 6: EXPORTAÇÃO E ANÁLISE FINAL
# ==============================================================================
print("\n>>> [FASE 6 de 6] Salvando os resultados...")

# Mostra a média REAL de cada grupo para você dar nomes a eles
perfil_real = df_final.groupby('cluster')[indicadores].mean().round(2)
print("\n--- PERFIL MÉDIO DE CADA GRUPO (VALORES REAIS) ---")
print(perfil_real)

# Salva o arquivo final para consulta
df_final.to_csv('resultado_segmentacao_final.csv', index=False)
print("\n>>> SUCESSO! Análise concluída. O arquivo 'resultado_segmentacao_final.csv' foi salvo.")